In [ ]:
import os
import json
import pandas as pd
import llm_blender
from itertools import combinations
from typing import List, Dict
from collections import defaultdict

blender = llm_blender.Blender()
blender.loadranker("llm-blender/PairRM")


def score_text(prompt: str, candidates: list[str]) -> dict[str, float]:
    n = len(candidates)
    if n == 0:
        return {}
    if n == 1:
        return {candidates[0]: 1.0}
    
    wins = {c: 0 for c in candidates}
    total = 0

    for a, b in combinations(candidates, 2):
        res = blender.compare([prompt], [a], [b])[0]
        wins[a] += int(res)
        wins[b] += int(not res)
        total += 1

    scores = {c: wins[c] / total for c in candidates}
    return scores


def load_subfolders(folder_path: str) -> list[str]:
    subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]
    return subfolders


def extract_seed(filename: str) -> int:
    seed_str = filename.replace('.json', '').replace('seed-', '')
    return int(seed_str)


def process_prompt_folders(folder_path: str) -> pd.DataFrame:
    all_results = []

    prompt_folders = load_subfolders(folder_path)
    
    print(f"Found {len(prompt_folders)} prompt folders")
    
    for prompt_folder in prompt_folders:

        prompt_data = []
        round_subfolders = load_subfolders(prompt_folder)
        
        for round_folder in round_subfolders:

            files = [os.path.join(round_folder, f) 
                    for f in os.listdir(round_folder) 
                    if f.endswith(".json")]
            
            for file in files:
                with open(file, "r") as f:
                    try:
                        data = json.load(f)
   
                        seed = extract_seed(os.path.basename(file))
                        data['seed'] = seed  
                        prompt_data.append(data)
                    except json.JSONDecodeError:
                        print(f"Could not parse {file}")
                        continue
        
        if not prompt_data:
            continue
            

        prompt = prompt_data[0]['prompt']
        
        response_to_data = {}
        unique_responses = []
        
        for item in prompt_data:
            response = item['response']
            if response not in response_to_data:
                response_to_data[response] = []
                unique_responses.append(response)
            response_to_data[response].append(item)
        
        if len(unique_responses) > 0:
            scores = score_text(prompt, unique_responses)

            for response, score in scores.items():
                for item in response_to_data[response]:
                    all_results.append({
                        'Prompt': item['prompt'],
                        'Response': item['response'],
                        'Model': item.get('model', 'unknown'),
                        'ID': item['seed'],
                        'Score': score
                    })
    
    df = pd.DataFrame(all_results)
    
    if not df.empty:
        df = df.sort_values(['Prompt', 'Score'], ascending=[True, False])
        df = df.reset_index(drop=True)
    
    return df


def main():    
    folder_path = "<folder_path_to_prompt_responses>"
    output_path = "<output_path_to_save_csv>"

    df = process_prompt_folders(folder_path)
    
    df.to_csv(output_path, index=False)
    
    return df


if __name__ == "__main__":
    df = main()

In [ ]:

average = df['Score'].mean()
print(f"Average Score: {average}")

Average Score: 0.07154414947797301
